# Tutorial 4: Extending Metric and Plot Components

Use the quickstart pipeline outputs to build custom metric and plot components with the current class-and-factory API. Custom classes can be supplied directly to a factory; reusable named components are supplied by installed plugins and resolved by name.

> **Notebook memory:** Importing the scientific Python stack (for example PyTorch, NumPy, Matplotlib, and ZenML) can keep approximately 1 GiB of RAM assigned to this kernel for its lifetime. Python cannot safely unload native extension modules. **After finishing this tutorial, restart its kernel to clear imported libraries and release that RAM** (or shut down the kernel entirely). Restarting keeps the notebook open with a fresh, low-memory kernel. Do this before running several tutorial notebooks at once.

In [ ]:
from collections.abc import Mapping
from pathlib import Path
from pprint import pprint
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch

from pioneerml.evaluation.metrics import BaseMetric, MetricFactory
from pioneerml.evaluation.plots import BasePlot, PlotFactory
from pioneerml.integration.zenml import load_step_output
from pioneerml.integration.zenml import utils as zenml_utils
from pioneerml.plugin import ensure_plugins_loaded

ensure_plugins_loaded()
from pioneerml_example_plugin.tutorial_examples.pipelines.quickstart_pipeline import quickstart_pipeline

zenml_client = zenml_utils.setup_zenml_for_notebook(use_in_memory=True)
print(f"ZenML initialized with stack: {zenml_client.active_stack_model.name}")

## Define custom metric and plot components

Components implement a small base-class contract:

- A metric subclasses `BaseMetric` and implements `compute(context=...)`.
- A plot subclasses `BasePlot` and implements `render(...)`.

Passing the class to its factory constructs it without registering notebook-local code globally. A production plugin would register a named class in its source package so configuration files could resolve it by name.

In [ ]:
class AverageTopConfidenceMetric(BaseMetric):
    """Measure the mean confidence of the highest-scoring class."""

    def compute(self, *, context: Mapping[str, Any]) -> dict[str, float]:
        predictions = torch.as_tensor(context["predictions"])
        probabilities = torch.sigmoid(predictions).detach().cpu()
        top_confidence = probabilities.max(dim=1).values.mean().item()
        return {"avg_top_confidence": float(top_confidence)}


class ClassFrequencyPlot(BasePlot):
    """Display the number of targets belonging to each class."""

    name = "class_frequency"

    def render(
        self,
        *,
        predictions,
        targets,
        class_names=None,
        save_path=None,
        show=False,
    ) -> str | None:
        predictions = torch.as_tensor(predictions)
        y_true = torch.as_tensor(targets).detach().cpu()
        if y_true.dim() == 1 and predictions.dim() == 2:
            if y_true.numel() % predictions.shape[-1] == 0:
                y_true = y_true.view(-1, predictions.shape[-1])
        if y_true.dim() == 1:
            y_true = y_true.unsqueeze(0)

        frequencies = y_true.sum(dim=0).numpy()
        labels = class_names or [str(i) for i in range(frequencies.shape[0])]

        fig, ax = plt.subplots(figsize=(6, 4))
        positions = np.arange(frequencies.shape[0])
        ax.bar(positions, frequencies)
        ax.set_xticks(positions)
        ax.set_xticklabels(labels)
        ax.set_ylabel("Count")
        ax.set_title("Target class frequency")
        fig.tight_layout()
        return self._finalize_figure(fig, save_path=save_path, show=show)


# Factories can construct an explicitly supplied class without global registration.
average_confidence_metric = MetricFactory(
    metric_cls=AverageTopConfidenceMetric,
).build()
class_frequency_plot = PlotFactory(
    plot_cls=ClassFrequencyPlot,
).build()

print(type(average_confidence_metric).__name__)
print(type(class_frequency_plot).__name__)

## Run quickstart pipeline and fetch outputs

The pipeline returns `(trained_module, datamodule, predictions, targets)`.

In [3]:
run = quickstart_pipeline.with_options(enable_cache=False)()
print(f"Pipeline run status: {run.status}")

trained_module = load_step_output(run, "train_module")
datamodule = load_step_output(run, "build_datamodule")
preds = load_step_output(run, "collect_predictions", output_name="output_0", index=0)
targets = load_step_output(run, "collect_predictions", output_name="output_1", index=0)

if preds is None or targets is None:
    outputs = load_step_output(run, "collect_predictions")
    if isinstance(outputs, (tuple, list)) and len(outputs) == 2:
        preds, targets = outputs

print("preds shape:", tuple(preds.shape) if preds is not None else None)
print("targets shape:", tuple(targets.shape) if targets is not None else None)


Initiating a new run for the pipeline: quickstart_pipeline.
Caching is disabled by default for quickstart_pipeline.
Using user: default
Using stack: default
  deployer: default
  artifact_store: default
  orchestrator: default
You can visualize your pipeline runs in the ZenML Dashboard. In order to try it locally, please run zenml login --local.
Step build_datamodule has started.
[build_datamodule] No materializer is registered for type <class 'pioneerml_example_plugin.tutorial_examples.pipelines.graph_datamodule.GraphDataModule'>, so the default Pickle materializer was used. Pickle is not production ready and should only be used for prototyping as the artifacts cannot be loaded when running with a different Python version. Please consider implementing a custom materializer for type <class 'pioneerml_example_plugin.tutorial_examples.pipelines.graph_datamodule.GraphDataModule'> according to the instructions at https://docs.zenml.io/concepts/artifacts/materializers
Step build_datamodule 

┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ DummyGraphClassifier │  630 K │ train │     0 │
│ 1 │ loss_fn │ BCEWithLogitsLoss     │      0 │ train │     0 │
└───┴─────────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 630 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 630 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 56                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

[train_module] /opt/conda/envs/pioneerml/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the num_workers argument to num_workers=11 in the DataLoader` to improve performance.

[train_module] /opt/conda/envs/pioneerml/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the num_workers argument to num_workers=11 in the DataLoader` to improve performance.

[train_module] Trainer.fit stopped: max_epochs=5 reached.


[train_module] No materializer is registered for type <class 'pioneerml.pipeline.services.training.utils.graph_lightning_module.GraphLightningModule'>, so the default Pickle materializer was used. Pickle is not production ready and should only be used for prototyping as the artifacts cannot be loaded when running with a different Python version. Please consider implementing a custom materializer for type <class 'pioneerml.pipeline.services.training.utils.graph_lightning_module.GraphLightningModule'> according to the instructions at https://docs.zenml.io/concepts/artifacts/materializers
Step train_module has finished in 2.908s.
Step collect_predictions has started.
Step collect_predictions has finished in 0.563s.
Pipeline run has finished in 5.525s.
Pipeline run status: completed
preds shape: (64, 3)
targets shape: (64, 3)


## Compute built-in and custom metrics

The built-in metric is resolved by its plugin name. The notebook-local metric is already constructed from its class. Both consume the same explicit context dictionary.

In [ ]:
probabilities = torch.sigmoid(preds)
metric_context = {
    "predictions": preds,
    "preds_binary": (probabilities >= 0.5).to(torch.float32),
    "targets": targets,
}

builtin_metric = MetricFactory(
    metric_name="binary_classification_from_tensors",
).build()

metrics = {}
for metric in (builtin_metric, average_confidence_metric):
    metrics.update(metric.compute(context=metric_context))

pprint(metrics)

## Build and display plots

Named built-in plots are resolved through `PlotFactory`. The notebook-local plot instance is called through the same `render(...)` interface. With `show=True`, `BasePlot` handles notebook display and figure cleanup consistently.

In [ ]:
loss_plot = PlotFactory(plot_name="loss_curves").build()
roc_plot = PlotFactory(plot_name="roc").build()
precision_recall_plot = PlotFactory(plot_name="precision_recall").build()
confusion_plot = PlotFactory(plot_name="multilabel_confusion").build()

loss_plot.render(
    getattr(trained_module, "train_epoch_loss_history", []),
    getattr(trained_module, "val_epoch_loss_history", []),
    title="Quickstart: Loss Curves",
    show=True,
)

plot_inputs = {
    "predictions": preds,
    "targets": targets,
    "class_names": ["pi", "mu", "e+"],
    "show": True,
}
roc_plot.render(**plot_inputs)
precision_recall_plot.render(**plot_inputs)
confusion_plot.render(**plot_inputs)
class_frequency_plot.render(**plot_inputs)